# Education 360° — Placement Analytics
Analyze placement outcomes, salary, company distribution, calls, course, and branch.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["braches dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses dim.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students dim.csv"]
}


def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

admissions      (2000, 10)
Could not find branches dim. Tried: ['branches_dim.csv', 'braches_dim.csv']. Expected dataset folder: c:\Users\aadit\Desktop\Data_Analyst_Portfolio\Project 03 — Academy Business Analytics\Dataset
calls           (18226, 7)
Could not find courses dim. Tried: ['courses_dim.csv']. Expected dataset folder: c:\Users\aadit\Desktop\Data_Analyst_Portfolio\Project 03 — Academy Business Analytics\Dataset
Could not find date table. Tried: ['DateTable.csv', 'date_table.csv']. Expected dataset folder: c:\Users\aadit\Desktop\Data_Analyst_Portfolio\Project 03 — Academy Business Analytics\Dataset
employees       (30, 5)
fees            (2000, 8)
leads           (2000, 6)
marketing       (150, 8)
placements      (2000, 7)
Could not find students dim. Tried: ['students_dim.csv']. Expected dataset folder: c:\Users\aadit\Desktop\Data_Analyst_Portfolio\Project 03 — Academy Business Analytics\Dataset


In [2]:
placements = datasets["placements"].copy()
students = datasets["students dim"].copy()
courses = datasets["courses dim"].copy()
branches = datasets["branches dim"].copy()
display(placements.head())

KeyError: 'students dim'

## Placement status and rate

In [ ]:
status = placements["placement status"].astype(str).str.strip()
status_summary = status.value_counts().rename_axis("Placement Status").reset_index(name="Students")
display(status_summary)
placed = (status == "Placed").sum()
total = placements["student id"].nunique()
print(f"Placed records: {placed:,}")
print(f"Distinct placement students: {total:,}")
print(f"Placement rate using placement table denominator: {placed/total*100:.2f}%")

## Salary analysis

In [ ]:
if "salary package lpa" in placements:
    placed_rows = placements.loc[status == "Placed"].copy()
    print("Average salary (LPA):", placed_rows["salary package lpa"].mean())
    display(placed_rows["salary package lpa"].describe())
    placed_rows["salary package lpa"].dropna().plot(kind="hist", bins=15, figsize=(8,4),
                                                    title="Salary Distribution — Placed Students")
    plt.xlabel("Salary package (LPA)"); plt.tight_layout(); plt.show()

## Placement by company

In [ ]:
if "company name" in placements:
    company_summary = placements.loc[status=="Placed"].groupby("company name").agg(
        Placed_Students=("student id","nunique"),
        Average_Salary=("salary package lpa","mean")
    ).reset_index().sort_values("Placed_Students", ascending=False)
    display(company_summary)

## Placement calls

In [ ]:
if "placement calls given" in placements:
    display(placements["placement calls given"].describe())
    print("Total placement calls:", placements["placement calls given"].sum())
    print("Students with exactly 5 calls:",
          (placements["placement calls given"] == 5).sum())

## Placement by course and branch

In [ ]:
student_course = students[["student id","course id","branch id"]].drop_duplicates()
p = placements.merge(student_course, on="student id", how="left")
if "course_id" in courses and "course_name" in courses:
    p = p.merge(courses[["course_id","course_name"]], left_on="course id", right_on="course_id", how="left")
if "branch id" in branches and "branch name" in branches:
    p = p.merge(branches[["branch id","branch name"]], on="branch id", how="left")
p["_placed"] = p["placement status"].astype(str).str.strip().eq("Placed")
for col in ["course_name","branch name"]:
    if col in p:
        display(p.groupby(col).agg(Students=("student id","nunique"),
                                   Placed=("_placed","sum")).assign(
            Placement_Rate_pct=lambda x: x["Placed"]/x["Students"]*100
        ).reset_index())

## Definition note
Keep the denominator explicit. The measure above uses distinct student IDs in the placements table; confirm whether the business definition should instead use all eligible students.